# TRACR Town05 Co-Simulation Attack Demo

This notebook mirrors the TRACR data-collection demo, but uses the Simu5G BSM stream and injects an obstacle ghost vehicle attack in front of the selected ego vehicle.

The CARLA scene uses Town05 from `configs/run_cosim_CARLAT5.json`. METS-R remains the traffic authority, CARLA visualizes the projected vehicles, and Simu5G carries the BSM stream. The attack sender is a fake static obstacle vehicle placed ahead of the ego vehicle. Its falsified position is redrawn in CARLA every BSM polling tick as an orange-red marker labeled `OBSTACLE GHOST BSM`.

## Setup

Load the same TRACR support utilities used by `tracr_demo.ipynb`, plus the obstacle ghost attack hook.

In [ ]:
from pathlib import Path
import importlib
import os
import sys

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == "tutorials":
    REPO_ROOT = REPO_ROOT.parent
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import tutorials.tracr_demo_support as tracr_demo_support
tracr_demo_support = importlib.reload(tracr_demo_support)

TRACRDashboard = tracr_demo_support.TRACRDashboard
launch_tracr_demo = tracr_demo_support.launch_tracr_demo
run_tracr_demo = tracr_demo_support.run_tracr_demo
benchmark_tracr_demo = tracr_demo_support.benchmark_tracr_demo
step_tracr_demo = tracr_demo_support.step_tracr_demo
configure_tracr_obstacle_ghost_attack = tracr_demo_support.configure_tracr_obstacle_ghost_attack

from cosim_utils.helpers import is_port_open
from utils.simu5g_v2x_util import start_simu5g_bridge_in_terminal

print("Repository root:", REPO_ROOT)

## Launch services

The original TRACR demo launches 200 private vehicles and marks the first 20 as C-V2X BSM emitters. This attack demo keeps the same scale, but uses `bsm_stream_source="simu5g"` so the dashboard BSM panel reflects messages delivered through the Simu5G bridge.

In [ ]:
VEINS_PORT = 9099
if not is_port_open(VEINS_PORT):
    print(f"Starting Simu5G bridge on port {VEINS_PORT}...")
    start_simu5g_bridge_in_terminal(REPO_ROOT, wait_seconds=5.0)

runtime = launch_tracr_demo(
    run_config="configs/run_cosim_CARLAT5.json",
    private_vehicle_count=200,
    v2x_vehicle_count=20,
    private_vehicle_start_id=1000,
    start_kafka=False,
    bsm_stream_source="simu5g",
    simu5g_host="127.0.0.1",
    simu5g_port=VEINS_PORT,
    require_simu5g_backend=True,
    start_metsr=True,
    start_carla=True,
    carla_camera_z=100.0,
    bsm_poll_timeout_ms=1,
    bsm_max_records=160,
    projection_heading_smoothing=0.35,
)

# Use the first connected vehicle as the demo ego/attack target.
runtime.focus_vehicle_id = runtime.v2x_vehicle_ids[0]
attack_config = configure_tracr_obstacle_ghost_attack(
    runtime,
    target_vehicle_id=runtime.focus_vehicle_id,
    ghost_vehicle_id=max(runtime.generated_vehicle_ids) + 1,
    distance_ahead_m=18.0,
    ghost_speed_mps=0.0,
    attack_id="tracr_purdue_obstacle_ghost",
)

{
    "viz_info": runtime.viz_info,
    "vehicle_count": len(runtime.generated_vehicle_ids),
    "v2x_vehicle_count": len(runtime.v2x_vehicle_ids),
    "ego_vehicle_id": runtime.focus_vehicle_id,
    "attack": attack_config,
}

## Demo dashboard

Open the printed local dashboard URL. The BSM panel shows the ego vehicle's received Simu5G BSM stream, including the injected obstacle ghost BSM when it is delivered. CARLA also shows an orange-red live marker at the falsified ghost position ahead of ego.

In [ ]:
dashboard = TRACRDashboard(
    stream_url=runtime.viz_info["url"],
    fullscreen=True,
    bsm_stream_label=runtime.bsm_stream_label,
    bsm_ego_only=True,
)
dashboard_url = dashboard.display_external(port=8899)
dashboard_url

## Run the live attack loop

This advances METS-R and CARLA, projects the Town05 traffic context into CARLA, synchronizes normal and ghost BSMs through Simu5G, and refreshes the dashboard. `bsm_every=1` keeps the orange-red ghost marker updated every loop iteration.

In [ ]:
last_result = benchmark_tracr_demo(
    runtime,
    dashboard,
    ticks=1500,
    sleep_s=0.0,
    render_wait_timeout=0,
    render_every=2,
    dashboard_every=3,
    bsm_every=1,
    sensor_every=1,
)
last_result["profile_summary_ms"]

## Manual step mode

Use this cell for a controlled walkthrough. Each call advances one synchronized step and redraws the obstacle ghost marker if the ego vehicle is live.

In [ ]:
step_result = step_tracr_demo(runtime, dashboard=dashboard, render_wait_timeout=0, poll_bsm=True)
{
    "ego_vehicle_id": getattr(runtime, "focus_vehicle_id", None),
    "last_obstacle_ghost": getattr(runtime, "tracr_last_obstacle_ghost", None),
    "bsm_count": len(step_result.get("bsm_records", [])),
    "step_result": step_result,
}

## Cleanup

Run this when the demo is done. This notebook does not start Kafka, so `stop_kafka=False` is appropriate.

In [ ]:
runtime.close(stop_kafka=False)